# First Parallel Scripts

The Python Virtual Machine is a scalar program environment. It has only one execution thread.

If you need to execute a parallel algorithm, you need to use specific modules.
There are several modules (you can consult those modules at the following link [https://wiki.python.org/moin/ParallelProcessing]).

The main problem is that most of the Parallel Modules are OS-dependent.

One of the most compatible modules is the multiprocessing module https://docs.python.org/3.11/library/multiprocessing.html



In [ ]:
from multiprocessing import Pool
import multiprocessing as mp
import numpy as np

To analyze our code we will import those two modules:
Time module, which allows us to calculate time, like the time spent to execute our code.
cProfile, which allows us to profile and analyze our code behaviour

In [ ]:
import time
import cProfile

## NEW FOR WINDOWS PLATFORM
In the following cell we import the module where we define the functions which we will execute in the parallel threads.

*NOTE*: The following lines are commented for the first test in Windows

In [ ]:
import myfunctions as my

Every time you make changes to the Python file *myfunctions.py*, you have to reload the module to update the changes.

In [ ]:
import importlib
importlib.reload(my)

## NUMBER OF CORES

In [ ]:
MULTITHREAD=True

In [ ]:
if MULTITHREAD:
    NUMCORES=int(mp.cpu_count()/2)
else:
    NUMCORES=int(mp.cpu_count())

In [ ]:
print(f"Number of cores available: {NUMCORES}")

# YOUR TASK (Next Session)
Analyze how to multiply two different matrices in parallel (the use of NumPy libraries is not allowed for this specific task).

Distribute both matrices (common data) and consider two possible approaches:
* Split and compute each cell independently.
* Split by rows and compute each row in parallel (with the elements of each row calculated sequentially).

Problems to solve:
* How to submit both matrices to the library for parallel processing.
* How to submit the coordinates for the cell or row to calculate.

In [ ]:
ROWS_A=250
COLS_A=325
ROWS_B=COLS_A
COLS_B=270

In [ ]:
A=np.random.random((ROWS_A,COLS_A))
B=np.random.random((ROWS_B,COLS_B))

In [ ]:
C_BASE= np.matmul(A,B)

In [ ]:
type(C_BASE)

## LINEAR CALCULATION

In [ ]:
def matmul_linear(matA: np.ndarray,
                  matB: np.ndarray) -> np.ndarray :
    a_rows, a_cols = matA.shape
    b_rows, b_cols = matB.shape
    C = np.zeros((a_rows,b_cols))
    for i in range(a_rows):
        for j in range(b_cols):
            for k in range (a_cols):
                C[i,j]+=matA[i,k]*matB[k,j]
    return C

In [ ]:
C_LINEAR=matmul_linear(A,B)

In [ ]:
np.allclose(C_BASE,C_LINEAR)

In [ ]:
%%timeit -r 1 -n 10
matmul_linear(A,B)

## Footprint Library Functions:

The functions in your library must have the following definitions:
<code>
    def init_pool(mA,mB)
    def cell_mult(coord: tuple) -> float
    def row_mult(coord: int) -> np.ndarray
</code>

## PARALLEL CALCULATION - FINE GRANULARITY

In [ ]:
from itertools import product

In [ ]:
def matmul_cell(matA: np.ndarray,
                matB: np.ndarray ) -> np.ndarray:
    a_rows, a_cols = matA.shape
    b_rows, b_cols = matB.shape
    C = np.zeros((a_rows,b_cols))
    coordenates = list(product(range(a_rows),range(b_cols)))
    with Pool(NUMCORES,my.init_pool,(matA,matB)) as p:
        r=p.map(my.cell_mult,coordenates)
    # Comment out the following 2 lines before execute the cell with timeit
    print(f"Type of r is: {type(r)}\n")
    print(f"Type of elements of r is: {type(r[0])}")
    
    for c in range(len(coordenates)):
        C[coordenates[c][0],coordenates[c][1]]=r[c]
    return C

In [ ]:
C_CELL=matmul_cell(A,B)

In [ ]:
np.allclose(C_BASE,C_CELL)

In [ ]:
%%timeit -r 1 -n 10
matmul_cell(A,B)

## PARALLEL CALCULATION - MEDIUM GRANULARITY

In [ ]:
def matmul_row(matA: np.ndarray,
                matB: np.ndarray ) -> np.ndarray:
    a_rows, a_cols = matA.shape
    b_rows, b_cols = matB.shape
    C = np.zeros((a_rows,b_cols))
    coordenates = list(range(a_rows))
    with Pool(NUMCORES,my.init_pool,(matA,matB)) as p:
        r=p.map(my.row_mult,coordenates)
    # Comment out the following 2 lines before execute the cell with timeit
    print(f"Type of r is: {type(r)}\n")
    print(f"Type of elements of r is: {type(r[0])}")
    for c in range(a_rows):
        C[c,:]=r[c]
    return C

In [ ]:
C_ROW=matmul_row(A,B)

In [ ]:
np.allclose(C_BASE,C_ROW)

In [ ]:
%%timeit -r 1 -n 10
matmul_row(A,B)

## Profiling
We will use the profiling tools to check our code behaviour

In [ ]:
cProfile.run("matmul_linear(A,B)")

In [ ]:
cProfile.run("matmul_cell(A,B)")

In [ ]:
cProfile.run("matmul_row(A,B)")